In [1]:
import sqlite3
import pandas as pd
import os

BASE = os.path.join(os.getcwd(), "..")  # project root
order_items = pd.read_csv(os.path.join(BASE, "Data", "raw", "order_items.csv"))
customers = pd.read_csv(os.path.join(BASE, "Data", "raw", "customers.csv"))
orders = pd.read_csv(os.path.join(BASE, "Data", "raw", "orders.csv"))
products = pd.read_csv(os.path.join(BASE, "Data", "raw", "products.csv"))
sellers = pd.read_csv(os.path.join(BASE, "Data", "raw", "sellers.csv"))
shipments = pd.read_csv(os.path.join(BASE, "Data", "raw", "shipments.csv"))


# Create in-memory database
conn = sqlite3.connect(":memory:")

# Load all tables
orders.to_sql("orders",           conn, index=False, if_exists="replace")
order_items.to_sql("order_items", conn, index=False, if_exists="replace")
customers.to_sql("customers",     conn, index=False, if_exists="replace")
shipments.to_sql("shipments",     conn, index=False, if_exists="replace")
products.to_sql("products",       conn, index=False, if_exists="replace")
sellers.to_sql("sellers",         conn, index=False, if_exists="replace")

# Run query
def run_query(sql):
    return pd.read_sql(sql, conn)

In [2]:
#B1. Monthly Marketplace Metrics
#Return, by calendar month and customer city: month, city, GMV, 
# number_of_orders, unique_customers,
#repeat_purchase_rate (customers with ≥2 delivered 
# orders up to that month / active customers in that month).



In [3]:
order_base = """
with cust_ord as (
    select orders.*,
           customers.city
    from orders 
    left join customers on orders.customer_id = customers.customer_id
),

base as (
    select *, 
           row_number() over(partition by customer_id order by created_at) as order_seq
    from cust_ord
),

delivered_base as (
    select *, 
           sum(case when status = 'Delivered' then 1 else 0 end) 
               over(partition by customer_id order by created_at 
                    rows between unbounded preceding and current row) as del_cumsum
    from base
),

v3 as (
    select *,
           concat(substring(created_at,1,4),'-',substring(created_at,6,2)) as year_month,
           case when order_seq = 1 and del_cumsum < 2 then 1 else 0 end as is_firsttime,
           case when del_cumsum >= 2 then 1 else 0 end as is_repeated
    from delivered_base
),

gmv as (
    select order_id, 
           sum(quantity * unit_price) as GMV 
    from order_items 
    group by order_id
)

select  
    v3.year_month,
    v3.city,
    count(distinct v3.order_id)                                           as total_orders,
    count(distinct v3.customer_id)                                        as unique_customers,
    count(distinct case when v3.is_firsttime = 1 then v3.customer_id end) as firsttime_cust,
    count(distinct case when v3.is_repeated  = 1 then v3.customer_id end) as repeated_cust,
    round(
        count(distinct case when v3.is_repeated = 1 then v3.customer_id end) * 100.0 / 
        count(distinct v3.customer_id), 2
    )                                                                     as repeat_purchase_rate,
    sum(gmv.GMV)                                                          as total_gmv

from v3 
join gmv on v3.order_id = gmv.order_id 
group by v3.year_month, v3.city
order by v3.year_month, v3.city;

"""

In [4]:
b1 = run_query(order_base)

In [5]:
#b1

In [6]:
#B2. Impact of First-Order Delay on Repeat
#For customers whose first delivered order was delayed (any Late bucket), 
# compare their 90-day repeat rate
#vs. customers whose first order was OnTime. Return a 
# table with two rows: first_order_delay_status =
#‘OnTime’ vs ‘Delayed’, and the corresponding 90-day repeat rate.


Step 1 → Find each customer's FIRST delivered order

Step 2 → Check if that first delivery was OnTime or Late

Step 3 → Check if they ordered again within 90 days

Step 4 → Group by OnTime vs Delayed → calculate repeat rate

In [7]:
b2 = """
with min_del_date as (
    select *,
           min(case when status = 'Delivered' then created_at else null end)
               over(partition by customer_id) as first_del_date
    from orders
),

is_first_del as (
    select *,
           case when first_del_date = created_at and status = 'Delivered' 
                then 1 else 0 
           end as is_first_del
    from min_del_date
),

first_del_orders as (
    select 
        i.customer_id,
        i.order_id,
        i.created_at as first_order_date,
        case when s.delivery_status = 'OnTime' then 'OnTime' 
             else 'Delayed' 
        end as first_order_delay_status
    from is_first_del i
    join shipments s on i.order_id = s.order_id
    where i.is_first_del = 1
),

repeat_check as (
    select 
        f.customer_id,
        f.first_order_date,
        f.first_order_delay_status,
        max(case when o.created_at > f.first_order_date
                 and julianday(o.created_at) - julianday(f.first_order_date) <= 90
                 and o.order_id != f.order_id
                 then 1 else 0 
            end) as is_repeat_90days
    from first_del_orders f
    left join orders o on f.customer_id = o.customer_id
    group by f.customer_id, f.first_order_date, f.first_order_delay_status
)

select 
    first_order_delay_status,
    count(customer_id) as total_customers,
    sum(is_repeat_90days) as repeat_customers,
    round(sum(is_repeat_90days) * 100.0 / count(customer_id), 2) as repeat_rate_90days
from repeat_check
group by first_order_delay_status
order by first_order_delay_status;
"""

In [8]:
run_query(b2)

,first_order_delay_status,total_customers,repeat_customers,repeat_rate_90days
0,Delayed,5762,2618,45.44
1,OnTime,17486,8088,46.25


#B3. Seller–Carrier Performance For (seller, carrier, ship_to_city) combinations with ≥100 delivered orders, return: seller_id, carrier,ship_to_city, total_GMV, delayed_GMV, delayed_order_rate, avg_delay_days (for delayed orders).

Step 1 → Join order_items + orders + shipments
         to get seller, carrier, city, GMV, delivery_status together

Step 2 → Filter only Delivered orders
         (brief says "delivered orders" for the ≥100 condition)

Step 3 → Calculate per (seller, carrier, ship_to_city):
         - total_GMV        → sum of GMV for all orders
         - delayed_GMV      → sum of GMV where delivery_status != OnTime
         - delayed_order_rate → delayed orders / total orders * 100
         - avg_delay_days   → avg diff_hours / 24 for delayed orders only

Step 4 → Filter combinations with ≥100 delivered orders

Step 5 → Sort by delayed_order_rate descending

In [9]:
b3 = """
WITH order_item AS (
    SELECT 
        order_id, 
        SUM(quantity * unit_price) AS total_gmv
    FROM order_items 
    GROUP BY order_id
),

base AS (
    SELECT 
        o.order_id,
        oi.total_gmv,
        s.carrier,
        s.ship_to_city,
        s.delivery_status
    FROM orders o
    JOIN order_item oi 
        ON o.order_id = oi.order_id
    JOIN shipments s 
        ON o.order_id = s.order_id
    WHERE o.status = 'Delivered'
)

SELECT 
    carrier,
    ship_to_city,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(total_gmv), 2) AS total_gmv,

    ROUND(SUM(
        CASE WHEN delivery_status != 'OnTime' 
             THEN total_gmv ELSE 0 END
    ), 2) AS delayed_gmv,

    ROUND(
        SUM(CASE WHEN delivery_status != 'OnTime' THEN 1 ELSE 0 END) * 100.0 
        / COUNT(DISTINCT order_id), 
    2) AS delayed_order_rate

FROM base
GROUP BY carrier, ship_to_city
HAVING COUNT(DISTINCT order_id) >= 100
ORDER BY delayed_order_rate DESC
"""

#b4 = """
#select distinct * from shipments
#"""

In [10]:
run_query(b3)

,carrier,ship_to_city,total_orders,total_gmv,delayed_gmv,delayed_order_rate
0,Delhivery,Jaipur,1372,45618910.0,41004445.0,88.92
1,Delhivery,Lucknow,1301,41264932.0,37845657.0,88.62
2,Ekart,Kolkata,1728,56414116.0,48668584.0,82.99
3,Ekart,Hyderabad,1590,57858668.0,23216160.0,28.05
4,Ekart,Bangalore,2285,76911471.0,27778875.0,27.88
5,Delhivery,Ahmedabad,1614,56999002.0,18321091.0,27.76
6,Ekart,Pune,1891,59311958.0,21283318.0,27.39
7,BlueDart,Kolkata,1748,59312805.0,22878090.0,27.35
8,Ekart,Lucknow,988,36291667.0,13295908.0,27.33
9,BlueDart,Ahmedabad,1197,40519134.0,13897640.0,27.32


In [11]:
b4_naive = """ 
 SELECT o.order_id,
(SELECT delivered_at
FROM shipments s
WHERE s.order_id = o.order_id) AS delivered_at,
o.created_at,
c.city
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE EXISTS (
SELECT 1
FROM shipments s2
WHERE s2.order_id = o.order_id
AND s2.delivery_status <> 'OnTime'
);


"""

problem_statement : Retrieve all orders that experienced delayed delivery (i.e., delivery_status ≠ 'OnTime'), along with their order date, delivery date, and customer city.

In [28]:
new_approach = """
with base_table as (

select distinct order_id,carrier, delivery_status from shipments where delivery_status <> 'OnTime') ,

order_table as (select orders.order_id, orders.customer_id,city,created_at, status,base_table.delivery_status, base_table.carrier
 from orders join base_table on base_table.order_id = orders.order_id left join 
 customers on customers.customer_id = orders.customer_id )

select * from order_table
"""

In [29]:
run_query(new_approach)

,order_id,customer_id,city,created_at,status,delivery_status,carrier
0,ORD000002,C12048,Kochi,2024-07-01,Delivered,Late_1_2d,Delhivery
1,ORD000012,C11631,Lucknow,2024-07-01,Delivered,Late_3_5d,Delhivery
2,ORD000016,C00794,Chennai,2024-07-01,Delivered,Late_1_2d,Ekart
3,ORD000018,C20816,Pune,2024-07-01,Delivered,Late_1_2d,Delhivery
4,ORD000022,C16339,Hyderabad,2024-07-01,Shipped,InTransit,BlueDart
...,...,...,...,...,...,...,...
28307,ORD099983,C00916,Bangalore,2025-12-30,Delivered,Late_1_2d,Delhivery
28308,ORD099984,C15302,Bangalore,2025-12-30,Delivered,Late_1_2d,BlueDart
28309,ORD099995,C00777,Delhi,2025-12-30,Shipped,InTransit,BlueDart
28310,ORD099996,C20036,Kolkata,2025-12-30,Returned,Late_1_2d,Delhivery


I replaced the correlated subquery with a CTE that precomputes delayed orders, reducing repeated scans. Then I used joins, which scale better. I’d also add indexes on order_id and delivery_status to speed up filtering and joins.”